# 🚀 LLaMA 7B Fine-tuning for Broking Domain

This notebook fine-tunes LLaMA 2 7B using Unsloth for stock market analysis and trade recommendations.

**Before running:**
1. Go to `Runtime` → `Change runtime type`
2. Select `T4 GPU` as Hardware accelerator
3. Click `Save`

---

## Step 1: Install Dependencies
This will take 2-3 minutes

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

# Disable wandb
import os
os.environ["WANDB_DISABLED"] = "true"

print("✅ Installation complete!")

## Step 2: Check GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3: Load Base Model
Loading LLaMA 2 7B with 4-bit quantization

In [ ]:
from unsloth import FastLanguageModel

# Configuration
max_seq_length = 2048
dtype = None  # Auto-detect
load_in_4bit = True  # Use 4-bit quantization

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-2-7b-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✅ Model loaded successfully!")

## Step 4: Add LoRA Adapters
Adding trainable adapters to the model

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✅ LoRA adapters added!")

## Step 5: Prepare Training Data
Broking domain training examples

In [ ]:
# Alpaca prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

# Training data for broking domain
training_data = [
    {
        "instruction": "Analyze the stock RELIANCE and provide a trading recommendation.",
        "input": "Current price: ₹2450, P/E ratio: 25.3, 52-week high: ₹2800, 52-week low: ₹2100, Volume: 5.2M shares",
        "output": "Based on the provided data for RELIANCE:\n\n**Technical Analysis:**\n- Current price (₹2450) is trading 12.5% below the 52-week high\n- Price is 16.7% above the 52-week low, showing good support\n- High trading volume indicates strong market interest\n\n**Fundamental Analysis:**\n- P/E ratio of 25.3 is reasonable for a diversified conglomerate\n\n**Recommendation: HOLD with BUY on dips**\n- Entry point: Consider accumulating below ₹2400\n- Stop loss: ₹2150\n- Target: ₹2700-2800\n\n*Disclaimer: This is not financial advice.*"
    },
    {
        "instruction": "What is the best time to buy stocks in the Indian market?",
        "input": "",
        "output": "The best time to buy stocks depends on several factors:\n\n**Market Timing:**\n1. Morning Session (9:15-10:00 AM): High volatility, good for experienced traders\n2. Mid-day (12:00-1:30 PM): Lower volatility, good for beginners\n3. Closing Hour (2:30-3:30 PM): Price discovery phase\n\n**Strategic Timing:**\n1. Market Corrections: Buy during 10-20% corrections\n2. Earnings Season: Post-results if fundamentals strong\n\nRemember: Time in the market beats timing the market."
    },
    {
        "instruction": "Explain what is a demat account and how to open one.",
        "input": "",
        "output": "A **Demat (Dematerialized) Account** holds your securities in digital format.\n\n**How to Open:**\n1. Choose a Depository Participant (broker like Zerodha, Groww)\n2. Submit documents: PAN, Aadhaar, Bank proof, Photos\n3. Complete e-KYC verification\n4. Account activated in 24-48 hours\n\n**Costs:**\n- Opening: Free to ₹500\n- Annual maintenance: ₹300-500\n- Transaction charges: ₹15-25"
    },
    {
        "instruction": "Provide a sector analysis for IT stocks in India.",
        "input": "Looking at major IT companies like TCS, Infosys, Wipro, HCL Tech",
        "output": "**Indian IT Sector Analysis**\n\n| Company | Strength | Rating |\n|---------|----------|--------|\n| TCS | Market leader | HOLD |\n| Infosys | Digital focus | BUY |\n| Wipro | Turnaround story | HOLD |\n| HCL Tech | Strong products | BUY |\n\n**Sector Drivers:**\n✅ Digital transformation demand\n✅ Cloud migration services\n\n**Headwinds:**\n⚠️ US recession concerns\n⚠️ Rising employee costs\n\n**Outlook: NEUTRAL to POSITIVE**"
    },
    {
        "instruction": "How do I place a stop-loss order?",
        "input": "",
        "output": "A **Stop-Loss Order** automatically sells when price falls to a specified level.\n\n**Steps:**\n1. Log into trading platform\n2. Select stock, choose SL or SL-M order\n3. Enter trigger price and limit price\n4. Submit order\n\n**Example:**\nBought HDFC Bank at ₹1600\n- Trigger: ₹1520 (5% below)\n- Limit: ₹1510\n\n**Best Practices:**\n- Set 5-10% below buy price\n- Don't set at obvious support levels"
    },
    {
        "instruction": "Analyze TATA Motors stock for investment.",
        "input": "Current price: ₹650, P/E: 8.5, Debt/Equity: 1.2, EV revenue growing 40% YoY",
        "output": "**TATA Motors Analysis**\n\n✅ **Strengths:**\n- Attractive P/E of 8.5\n- EV segment growing 40% YoY\n- Strong domestic market share\n\n⚠️ **Concerns:**\n- Debt/Equity of 1.2 is elevated\n- JLR exposure to global recession\n\n**Recommendation: BUY**\n- Entry: ₹630-650\n- Stop Loss: ₹560\n- Target: ₹720-800\n\n**Risk Level: MEDIUM-HIGH**"
    },
    {
        "instruction": "What is intraday trading vs delivery trading?",
        "input": "",
        "output": "**Intraday vs Delivery Trading:**\n\n| Feature | Intraday | Delivery |\n|---------|----------|----------|\n| Holding | Same day | Unlimited |\n| Leverage | 5x-20x | None |\n| Risk | Very High | Moderate |\n| STT | 0.025% | 0.1% |\n\n**Choose Intraday if:** Can monitor markets, experienced\n**Choose Delivery if:** Long-term investing, lower risk\n\n**Warning:** 90% of intraday traders lose money."
    },
    {
        "instruction": "Give me a recommendation for banking sector stocks.",
        "input": "I want exposure to Indian banking sector with moderate risk",
        "output": "**Banking Sector Recommendations:**\n\n1. **HDFC Bank** - ₹1650, Target ₹1850 (LOW risk)\n2. **ICICI Bank** - ₹1050, Target ₹1200 (LOW-MEDIUM)\n3. **SBI** - ₹650, Target ₹750 (MEDIUM)\n4. **Kotak Bank** - ₹1750, Target ₹2000 (MEDIUM)\n\n**Suggested Allocation:**\n- HDFC: 35%\n- ICICI: 30%\n- SBI: 20%\n- Kotak: 15%\n\n**Investment Horizon:** 1-3 years"
    },
    {
        "instruction": "What is an IPO and how do I apply?",
        "input": "",
        "output": "**IPO (Initial Public Offering)**\n\nWhen a private company offers shares to public for the first time.\n\n**How to Apply:**\n1. Log into broker app\n2. Go to IPO section\n3. Select IPO, enter lots\n4. Approve UPI mandate\n5. Amount blocked until allotment\n\n**Categories:**\n- Retail: Up to ₹2 lakh\n- HNI: Above ₹2 lakh\n\n**Tip:** Check company fundamentals and GMP before applying."
    },
    {
        "instruction": "Hello, can you help me with stock market questions?",
        "input": "",
        "output": "Hello! Of course, I'd be happy to help you with any stock market questions.\n\nI can assist with:\n- Stock analysis and recommendations\n- Trading strategies\n- IPOs and corporate actions\n- Account and procedures\n- Market concepts\n\nWhat would you like to know about?"
    },
    {
        "instruction": "What are trading charges in Indian stock market?",
        "input": "",
        "output": "**Stock Market Charges:**\n\n1. **Brokerage:** ₹20/trade or 0.03%\n2. **STT:** 0.1% (delivery), 0.025% (intraday)\n3. **Exchange charges:** 0.003%\n4. **GST:** 18% on brokerage\n5. **Stamp Duty:** 0.015%\n6. **DP charges:** ₹13-18 per sell\n\n**Example (₹1L delivery trade):**\nTotal charges ≈ ₹142\n\n**Tip:** Use discount brokers to reduce costs."
    },
    {
        "instruction": "Analyze HDFC Bank stock.",
        "input": "Price: ₹1650, P/E: 18.5, NPA: 1.2%, ROE: 16.5%",
        "output": "**HDFC Bank Analysis**\n\n✅ **Strengths:**\n- P/E of 18.5 reasonable for quality\n- Excellent NPA of 1.2%\n- Strong ROE of 16.5%\n\n⚠️ **Concerns:**\n- Merger integration ongoing\n- FII selling pressure\n\n**Recommendation: ACCUMULATE**\n- Entry: ₹1600-1650\n- Stop Loss: ₹1500\n- Target: ₹1800-1950\n\n**Verdict:** Core portfolio holding for conservative investors."
    }
]

print(f"✅ Loaded {len(training_data)} training examples")

In [ ]:
from datasets import Dataset

# Format data with Alpaca template
def format_example(example):
    text = alpaca_prompt.format(
        instruction=example["instruction"],
        input=example["input"],
        output=example["output"],
    )
    return {"text": text}

formatted_data = [format_example(ex) for ex in training_data]
dataset = Dataset.from_list(formatted_data)

print(f"✅ Dataset prepared with {len(dataset)} examples")
print(f"\nSample formatted prompt:\n{dataset[0]['text'][:500]}...")

## Step 6: Train the Model
This will take ~5-10 minutes on T4 GPU

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir="./outputs",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=1,
        save_steps=50,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()

print(f"\n✅ Training completed!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## Step 7: Test the Model
Let's test our fine-tuned model!

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

# Test prompt
test_prompt = alpaca_prompt.format(
    instruction="Recommend a good stock for long-term investment.",
    input="I have moderate risk appetite and looking for 2-3 year horizon.",
    output="",
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=" * 50)
print("MODEL RESPONSE:")
print("=" * 50)
print(response.split("### Response:")[-1].strip())

In [ ]:
# Test another prompt
test_prompt2 = alpaca_prompt.format(
    instruction="What is F&O trading? Should beginners try it?",
    input="",
    output="",
)

inputs = tokenizer(test_prompt2, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=" * 50)
print("MODEL RESPONSE:")
print("=" * 50)
print(response.split("### Response:")[-1].strip())

## Step 8: Save Model & Download to Your Computer
Save model locally and download it

In [ ]:
import os

# Create local output directory
save_path = "./broking-llama-7b"
os.makedirs(save_path, exist_ok=True)

print(f"Saving model to: {save_path}")

In [ ]:
# Save LoRA adapters (small, ~50MB)
lora_path = f"{save_path}/lora_adapters"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"✅ LoRA adapters saved to: {lora_path}")

In [ ]:
# Save to GGUF format (for llama.cpp/Ollama deployment)
# This creates a quantized model that runs on CPU too!

print("Converting to GGUF format (this may take a few minutes)...")
model.save_pretrained_gguf(
    save_path,
    tokenizer,
    quantization_method="q4_k_m"  # Good balance of quality and size (~4GB)
)
print(f"✅ GGUF model saved!")

In [ ]:
# List saved files
import os
print("\n📁 Saved files:")
for root, dirs, files in os.walk(save_path):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath) / (1024*1024)  # MB
        print(f"  {file}: {size:.1f} MB")

In [ ]:
# Zip the LoRA adapters for easy download
!cd broking-llama-7b && zip -r ../lora_adapters.zip lora_adapters/
print("✅ Created lora_adapters.zip")

In [ ]:
# Download files to your local computer
from google.colab import files

print("📥 Downloading LoRA adapters (small, ~50MB)...")
files.download('lora_adapters.zip')

In [ ]:
# Download GGUF file (larger, ~4GB)
# Find the GGUF file
import glob
gguf_files = glob.glob(f"{save_path}/*.gguf")
if gguf_files:
    gguf_file = gguf_files[0]
    print(f"📥 Downloading GGUF file: {gguf_file}")
    print("⚠️ This is a large file (~4GB), download may take a while...")
    files.download(gguf_file)
else:
    print("No GGUF file found")

## ✅ Done!

Your fine-tuned model is ready for download!

**Downloaded files:**
- `lora_adapters.zip` - LoRA adapters (~50MB) - use with transformers
- `*.gguf` - Quantized model (~4GB) - use with Ollama/llama.cpp

**Next Steps for GGUF (Ollama):**
1. Install Ollama: https://ollama.ai
2. Create a Modelfile:
```
FROM ./broking-llama-7b-q4_k_m.gguf
TEMPLATE """### Instruction:\n{{.Prompt}}\n\n### Response:\n"""
```
3. Run: `ollama create broking-llama -f Modelfile`
4. Chat: `ollama run broking-llama`